<a href="https://colab.research.google.com/github/Yashraj-mlrobo/flyrank-task-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yashraj-mlrobo/flyrank-task-1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
''' One row represents one unique content_id evaluated during a specific tracking month (month = '2026-03'). '''
"Evaluated over a specific mid-panel tracking month (month = '2026-03'), leaving the final month (June 2026) sealed as the future outcome window."
from google.colab import userdata
from datasets import load_dataset

hf_token = userdata.get('HF_TOKEN')


dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files="fact_content_daily_performance/month=2026-03/*.parquet",
    token=hf_token
)

df = dataset['train'].to_pandas()
print(f"Loaded mid-panel month successfully! Rows: {len(df)}")

# Verify exact date range of the slice
print(f"Date Span: {df['report_date'].min()} to {df['report_date'].max()}")

#  Verify grain uniqueness (Checking for duplicate entries on the same day)
duplicates = df.duplicated(subset=['content_hash_id', 'report_date']).sum()
print(f"Duplicate content_id per day count: {duplicates} (Expected: 0)")

Loaded mid-panel month successfully! Rows: 9841378
Date Span: 2026-03-01 to 2026-03-31
Duplicate content_id per day count: 0 (Expected: 0)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
"""
Feature:

    gsc_avg_position: Used to calculate position gap relative to baseline (gsc_avg_position - 0.55).

    gsc_impressions & gsc_clicks: Used to compute baseline click-through rates (feat_ctr_calc).

    ga4_sessions & sessions_organic: Used to determine organic traffic ratios (feat_organic_ratio).

    ga4_total_engagement_sec: Used to measure user engagement per session (feat_engagement_rate).

    sessions_ai: Used to calculate AI-referred traffic share (feat_ai_share).

Label / Proxy:

    target_degraded: Binary opportunity flag (derived dynamically by identifying content pieces falling into the bottom 25th percentile of historical baseline CTR).

Context:

    report_date: Evaluation timestamp defining the historical tracking window (March 2026).

    content_hash_id: Unique content identifier defining the grain of analysis.

    client_hash_id: Client group identifier for organizational partitioning.

Excluded (with justification):

    gsc_data_available & ga4_data_available: Why: Used strictly as row availability filters (IS TRUE) during preprocessing; excluding raw boolean flags avoids redundant zero-variance columns in the feature matrix.

    ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other: Why: Individual referral breakdown flags introduce high sparsity and noise; aggregated into the robust top-level sessions_ai feature instead.

    scroll_events: Why: Inconsistent tracking implementations across client domains lead to missing values, making it an unreliable signal for automated scoring.

"""

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# ==============================================================================
# CLAIM 1: Unit of Analysis & Grain (1 row = 1 unique content_hash_id per report_date)
# ==============================================================================
grain_duplicates = df.duplicated(subset=['content_hash_id', 'report_date']).sum()
print(f"[VERIFIED] Claim 1 (Grain): Duplicate rows = {grain_duplicates} (Expected: 0)")

[VERIFIED] Claim 1 (Grain): Duplicate rows = 0 (Expected: 0)


In [7]:
# ==============================================================================
# CLAIM 2: Time Window & Row Counts (Mid-panel month = 2026-03)
# ==============================================================================
min_date = df['report_date'].min()
max_date = df['report_date'].max()
total_rows = len(df)
print(f"[VERIFIED] Claim 2 (Time Window): Date Range = {min_date} to {max_date}")
print(f"[VERIFIED] Claim 2 (Row Count): Total Rows Loaded = {total_rows}")

[VERIFIED] Claim 2 (Time Window): Date Range = 2026-03-01 to 2026-03-31
[VERIFIED] Claim 2 (Row Count): Total Rows Loaded = 9841378


In [8]:
# ==============================================================================
# CLAIM 3: Availability Filtering (gsc_data_available IS TRUE)
# ==============================================================================
# Filter with IS TRUE and measure surviving rows
surviving_mask = (df['gsc_data_available'] == True)
surviving_rows = surviving_mask.sum()
surviving_pct = (surviving_rows / total_rows) * 100

print(f"[VERIFIED] Claim 3 (Availability IS TRUE): {surviving_rows:,} / {total_rows:,} rows survive ({surviving_pct:.2f}%)")

# Filter dataframe to active slice for downstream feature extraction
df_active = df[surviving_mask].copy()

[VERIFIED] Claim 3 (Availability IS TRUE): 3,611,061 / 9,841,378 rows survive (36.69%)


In [9]:
# ==============================================================================
# CLAIM 4: Missing Values Audit (Checking nulls across key feature columns)
# ==============================================================================
check_cols = ['gsc_avg_position', 'gsc_clicks', 'gsc_impressions', 'ga4_sessions', 'sessions_organic', 'sessions_ai']
missing_summary = df_active[check_cols].isnull().sum()

print("\n[VERIFIED] Claim 4 (Missing Values Audit in Active Slice):")
print(missing_summary)


[VERIFIED] Claim 4 (Missing Values Audit in Active Slice):
gsc_avg_position          0
gsc_clicks                0
gsc_impressions           0
ga4_sessions        1528366
sessions_organic    1528366
sessions_ai         1528366
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [10]:
"""
Off-Page & Offline Factors: The dataset tracks on-page SEO metrics (GSC/GA4) but is completely blind to external search drivers like backlink profile changes, competitor activity, brand marketing campaigns, or Google core algorithm updates occurring within the window.

GSC-Only Early Rows & Tracking Asymmetry: Days where gsc_data_available IS TRUE but ga4_data_available IS FALSE create asymmetric histories. Metrics relying on GA4 sessions or engagement will be missing or artificially zeroed out for those early periods without reflecting true user engagement.

Unbalanced History & Blind Spots for New Content: Because the slice filters for established performance thresholds, newly published pages or URLs without historical GSC impressions are completely omitted. The model cannot score or evaluate the refresh potential of cold-start content.

Window Overlaps & Attribution Ambiguity: Aggregating performance over fixed monthly windows (report_date) obscures intra-month volatility. If a content update occurred mid-month, the dataset cannot isolate pre- versus post-refresh metrics within that same tracking window, creating attribution overlap.

"""

'\nOff-Page & Offline Factors: The dataset tracks on-page SEO metrics (GSC/GA4) but is completely blind to external search drivers like backlink profile changes, competitor activity, brand marketing campaigns, or Google core algorithm updates occurring within the window.\n\nGSC-Only Early Rows & Tracking Asymmetry: Days where gsc_data_available IS TRUE but ga4_data_available IS FALSE create asymmetric histories. Metrics relying on GA4 sessions or engagement will be missing or artificially zeroed out for those early periods without reflecting true user engagement.\n\nUnbalanced History & Blind Spots for New Content: Because the slice filters for established performance thresholds, newly published pages or URLs without historical GSC impressions are completely omitted. The model cannot score or evaluate the refresh potential of cold-start content.\n\nWindow Overlaps & Attribution Ambiguity: Aggregating performance over fixed monthly windows (report_date) obscures intra-month volatility. 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.